# Sweden road/rail network-tier barrier matching — verifying `schools/assemble.py`'s `same_route`/`same_side` output

**Scope.** This notebook verifies the *network-aware* matching tiers only
(`schools/assemble.py` algorithms 4–5: `same_route`/`same_side`, using
`road_network`/`network` (rail) as candidate geometry) against their real,
persisted output — including the road `same_side` data-provenance bug
(barrier geometry coincides with the road centerline, collapsing
`same_side` to near-zero) and whether the barrier's own `side` attribute
could fix it. **What point/`same_route`/`same_side` mean in general, and
why each tier should nest inside the last, is cross-region** — see
[`shared_methodology.ipynb` §1](../shared_methodology.ipynb); Florida runs
the same three tiers against its own arterial network in
[`florida/schools.ipynb`](../florida/schools.ipynb).

**Elsewhere.**
- Raw barrier-register descriptives, cleaning, and the simple point-distance
  match (algorithms 1–2) live in [`barriers.ipynb`](barriers.ipynb).
- Whether `skolenhetskod` itself is stable across school reorganizations,
  its panel impact, and vanished pre-registry schools live in
  [`schools_lineage.ipynb`](schools_lineage.ipynb).


> **Note (2026-09-23): historical record.** The network-matching cells
> (the `wall_side` / `school_side` analysis in §3 and after) read the
> `schools_*_pairs_network.parquet` schema from *before* the barrier-matching
> rewrite. That schema had `school_side`/`wall_side` columns; the current
> one has `side_method`/`same_side`/`same_side_unknown` instead. The data
> they ran on is kept in `data/sweden/_backup_pre_barrier_reference_2026-09-23/`.
> The findings recorded here were re-examined with better ground truth in
> [`docs/data/sweden/barrier_matching.md`](../../../docs/data/sweden/barrier_matching.md):
> the road offset really is degenerate, but so is rail's (99.7% of rail
> barriers sit on the track), and the `side` attribute is still unusable.

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "src").is_dir() and (c / "data").is_dir():
            return c
    raise FileNotFoundError(f"repo root not found from {Path.cwd()}")


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiments._setup import BLUE, GREEN, GREY, RED, np, pd, plt

DATA = ROOT / "data" / "sweden"

print("root:", ROOT)

import geopandas as gpd

SCHOOLS = DATA / "schools"
print(sorted(p.name for p in (SCHOOLS / "processed").glob("*")))
print(sorted(p.name for p in (SCHOOLS / "assembled").glob("*.parquet")))


root: /Users/felixschulz/Library/CloudStorage/OneDrive-Personal/Dokumente/Job/UNI/Basel/Research/noise-pollution


['schools.csv', 'schools.geojson', 'skolenhetskod_lineage.csv']
['schools_barrier_rollup.parquet', 'schools_rail_pairs.parquet', 'schools_rail_pairs_network.parquet', 'schools_rail_rollup.parquet', 'schools_rail_rollup_network.parquet', 'schools_road_pairs.parquet', 'schools_road_pairs_network.parquet', 'schools_road_rollup.parquet', 'schools_road_rollup_network.parquet']


## 1. Algorithms 4–5 (`same_route` / `same_side`) — verify the tiers actually nest

Each tier should be a strict refinement of the last: `same_side ⊆
same_route ⊆ point`.


In [2]:
def tier_counts(kind: str) -> pd.Series:
    rollup = pd.read_parquet(SCHOOLS / "assembled" / f"schools_{kind}_rollup_network.parquet")
    counts = pd.Series({
        "point": int(rollup["ever_treated"].sum()),
        "same_route": int(rollup["ever_treated_same_route"].sum()),
        "same_side": int(rollup["ever_treated_same_side"].sum()),
    })
    # nesting check: every same_route school must also be ever_treated (point), etc.
    assert set(rollup.loc[rollup["ever_treated_same_route"], "skolenhetskod"]) <= set(
        rollup.loc[rollup["ever_treated"], "skolenhetskod"])
    assert set(rollup.loc[rollup["ever_treated_same_side"], "skolenhetskod"]) <= set(
        rollup.loc[rollup["ever_treated_same_route"], "skolenhetskod"])
    return counts

tiers = pd.DataFrame({kind: tier_counts(kind) for kind in ("road", "rail")})
print(tiers)
print("\nNesting holds for both kinds (assertions passed).")


            road  rail
point       1251  1454
same_route   967  1059
same_side      4   749

Nesting holds for both kinds (assertions passed).


Rail collapses gently tier by tier (a real refinement); road's `same_side`
column collapses almost to nothing — reproduced live below, matching the
README's "not a matching bug, a real data-provenance finding".


## 2. Road `same_side` is broken — confirmed live, not just asserted from the doc


In [3]:
road_pairs = pd.read_parquet(SCHOOLS / "assembled" / "schools_road_pairs_network.parquet")
same_route_pairs = road_pairs[road_pairs["same_route"].fillna(False)]
on_centerline = (same_route_pairs["wall_side"] == 0)
print(f"{len(same_route_pairs):,} road same_route pairs; "
      f"{on_centerline.sum():,} ({on_centerline.mean():.1%}) have wall_side == 0 (on the road centerline)")

rail_pairs = pd.read_parquet(SCHOOLS / "assembled" / "schools_rail_pairs_network.parquet")
rail_same_route = rail_pairs[rail_pairs["same_route"].fillna(False)]
print("\nrail wall_side value counts among same_route pairs (for comparison — a real 3-way split expected):")
print(rail_same_route["wall_side"].value_counts())


4,610 road same_route pairs; 4,603 (99.8%) have wall_side == 0 (on the road centerline)

rail wall_side value counts among same_route pairs (for comparison — a real 3-way split expected):
wall_side
 1.0    2645
-1.0    2331
 0.0    1225
Name: count, dtype: int64


Road barrier geometry is (near-)coincident with the road centerline
itself — `wall_side` is ≈0 for the overwhelming majority of same-route
pairs, so the side comparison measures noise, not a real left/right
distinction, for road specifically. Rail's side field has a genuine 3-way
split, confirming rail barrier geometry *is* independently surveyed and
the bug is road-specific, not a shared implementation error.
**Conclusion, matching the README: use `ever_treated_same_route` for road,
not `ever_treated_same_side`.**


## 3. Can the barrier's own `side` attribute resolve road's `same_side`?

`wall_side` is *derived* from barrier geometry, which is degenerate for
road (§4). But `noise_barriers/preprocess.py` also carries the barrier's
own recorded `side` field — Trafikverket's own left/right siting
attribute, not something this pipeline computes. If that field is
reliable, it sidesteps the degenerate-geometry problem entirely: compare
the barrier's recorded `side` (converted to the same signed convention)
against the school's own geometric `school_side` (which is NOT degenerate
— schools are real points, not collinear with the road), instead of
re-deriving `wall_side` from barrier geometry at all. The check below
starts with the one road example available, then validates the idea
properly at scale using rail (where geometry is *not* degenerate) as an
independent ground truth for the same field.


In [4]:
NOISE_BARRIERS = DATA / "noise_barriers"
road_barriers = gpd.read_parquet(NOISE_BARRIERS / "processed" / "road_noise_barriers.parquet").reset_index(names="barrier_row")
print("`side` field population:", road_barriers["side"].notna().mean(), "of", len(road_barriers), "road barriers")
print(road_barriers["side"].value_counts(dropna=False))

# The only real (non-degenerate) geometric `wall_side` values among road
# same_route pairs (§4's 0.2% off-centerline remainder) -- cross-check
# whether the barrier's recorded `side` is internally consistent with the
# sign `_linear_ref.py::position_and_side` independently derived from
# geometry for that same barrier.
nonzero = same_route_pairs[same_route_pairs["wall_side"] != 0].merge(
    road_barriers[["barrier_row", "side"]], on="barrier_row"
)
print(f"\n{len(nonzero)} same_route pairs with a real (non-centerline) geometric wall_side, "
      f"from {nonzero['barrier_row'].nunique()} distinct barrier(s):")
print(nonzero[["barrier_row", "wall_side", "side"]].drop_duplicates())


`side` field population: 1.0 of 2172 road barriers
side
right         1732
left           436
both_sides       4
Name: count, dtype: int64

7 same_route pairs with a real (non-centerline) geometric wall_side, from 1 distinct barrier(s):
   barrier_row  wall_side   side
0          442       -1.0  right


**`side` is 100% populated** (`right`/`left`/`both_sides`), unlike the
degenerate geometry. The one road cross-check available (the sole
barrier whose geometry isn't exactly on the centerline) happened to line
up with `side` under a fixed sign convention — but **this is N=1, not a
real validation**: every one of the other 4,602 on-centerline pairs
shares the same degenerate-geometry problem, so there's no independent
geometric signal left to check `side` against for road specifically.

### Validating `side` at scale, using rail as an independent check

Rail barriers carry the exact same NVDB `side` field (same
`noise_barriers` product, same schema) — but rail's `wall_side` is *not*
degenerate: §4 already showed a real 3-way geometric split
(2645/−2331/1225) for rail. That makes rail a genuine, large-N test of
whether `side`'s left/right convention actually predicts the
independently-surveyed geometric side at all — a proxy for whether the
same field would be trustworthy for road, without needing a second
non-degenerate road barrier that doesn't exist in this data.


In [5]:
rail_barriers = gpd.read_parquet(NOISE_BARRIERS / "processed" / "rail_noise_barriers.parquet").reset_index(names="barrier_row")
rail_pairs = pd.read_parquet(SCHOOLS / "assembled" / "schools_rail_pairs_network.parquet")
rail_same_route = rail_pairs[rail_pairs["same_route"].fillna(False)]
rail_nonzero = rail_same_route[rail_same_route["wall_side"] != 0].merge(
    rail_barriers[["barrier_row", "side"]], on="barrier_row"
)

per_barrier = rail_nonzero.groupby("barrier_row").agg(
    wall_side=("wall_side", "first"),
    wall_side_nunique=("wall_side", lambda s: s.round(0).nunique()),
    side=("side", "first"),
)
print(f"{len(rail_nonzero):,} rail same_route pairs with non-centerline wall_side, "
      f"from {per_barrier.shape[0]:,} distinct barriers")
print("barriers with an internally inconsistent wall_side across their own pairs:",
      int((per_barrier["wall_side_nunique"] > 1).sum()),
      "(0 expected -- wall_side should be a barrier-level property, not vary pair to pair)")
print()
print("`side` vs. geometric `wall_side`, one row per barrier:")
print(pd.crosstab(per_barrier["side"], per_barrier["wall_side"]))


4,976 rail same_route pairs with non-centerline wall_side, from 949 distinct barriers
barriers with an internally inconsistent wall_side across their own pairs: 0 (0 expected -- wall_side should be a barrier-level property, not vary pair to pair)

`side` vs. geometric `wall_side`, one row per barrier:
wall_side   -1.0   1.0
side                  
both_sides     0     1
center         1     1
left         227   245
right        228   246


**Decisive negative result — the fix does not work.** Across 949 rail
barriers with a real (non-centerline) geometric `wall_side`,
`side="left"` splits 227/245 and `side="right"` splits 228/246 between
the two geometric signs — a coin flip, not a correlation. `wall_side` is
confirmed stable per barrier (0 inconsistencies across pairs), so this
isn't a matching artifact: `side` genuinely carries no usable signal
about the geometrically-derived side, at least not under any fixed sign
convention. The N=1 road match in the cell above was coincidental, not
evidence the field is usable — with a 50/50 base rate, a single match is
exactly what pure chance predicts.

Two candidate explanations were checked and neither rescues the idea:

- **Digitizing-direction inconsistency** — the exact risk
  `_linear_ref.py`'s own docstring flags, and the failure mode Florida
  found in `rciroads`. If segments were stored with mixed direction
  order, `position_and_side`'s sign would flip unpredictably relative to
  any fixed real-world reference, which could produce exactly this kind
  of coin-flip. But checking `network_tracks.parquet` directly:
  `start_measure < end_measure` holds for 100% of segments (tautological
  by construction) and `km_from_m < km_to_m` holds for 98.8% — the
  network's own bookkeeping shows almost no reversed segments, so this
  doesn't explain the result either.
- `side_code` (the field's numeric encoding) is a redundant duplicate of
  `side`, sometimes internally inconsistent with it — no additional
  signal to recover there.

**Conclusion: `side` cannot fix road's `same_side`.** It's not a null
field (100% populated) and not a matching bug — it simply does not
correlate with the independently-computed geometric side at any usable
rate, even on rail where the geometry itself is not degenerate.
**Recommendation: keep using `ever_treated_same_route` for road;
`ever_treated_same_side` stays broken/unusable for road, with no
attribute-based substitute found.** Building the previously proposed
`school_side == sign(side)` fix would inject noise, not signal, into
road's treatment definition — no pipeline change follows from this
result.


## Summary

- The three network-aware tiers nest as designed for both road and rail
  (checked via set-containment assertions against the real persisted
  rollup, not just by eyeballing counts).
- Road's `same_side` is confirmed broken live from the actual pair table
  (near-100% centerline coincidence), not just quoted from the README —
  `ever_treated_same_route` is the right road-network-refined treatment
  set to use, matching `panel/assemble.py`'s own six parallel definitions.
- **The barrier's own `side` attribute does NOT fix road `same_side`**:
  validated at scale using rail (949 barriers with real, non-degenerate
  geometric `wall_side`), `side` splits ~50/50 against the independently
  computed geometric side for both `left` and `right` — no usable
  correlation. The earlier N=1 road cross-check was coincidental, not
  evidence. Digitizing-direction inconsistency (the risk
  `_linear_ref.py` itself warns about) was checked and ruled out as the
  cause. No pipeline change follows: continue using
  `ever_treated_same_route` for road; `ever_treated_same_side` remains
  broken/unusable with no attribute-based substitute found.
